# RunPod: the flat model (level 1), F1 and its planning

Run the cells from the top, on an A100 pod (about $1.8/h). Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC`, then open this notebook from there. The dataset comes from the Hub repo that `runpod_dataset.ipynb` filled. The helpers are in `notebooks/pod.py` (shared with the variant notebooks `runpod_V*_*.ipynb`, one per variant).

The network volume at `/workspace` refuses `chmod` (`git clone`, `venv` and `pip` fail on it) and stalls on the random reads of the training file, so only what must survive a pod stop lives there:

| what | where | after a pod restart |
|---|---|---|
| repository, venv | local disk: `/root/AAC`, `/root/venvs/hjepa` | clone by hand, the venv installs again (a few minutes) |
| episode files | local disk: `/root/data`, downloaded from the Hub (about 9 GB) | downloaded again |
| checkpoints, reports, task files | the volume: `/workspace/data`, linked into `/root/data` | kept, F1 resumes |
| logs (`/root/logs`), stable-pretraining's folder (`/root/spt`) | local disk: written at every logging step, which stalled the training about 10 s each on the volume | lost, nothing needed for the resume |

| step | what |
|---|---|
| 0 | configuration, Hugging Face login, the PyTorch environment, the dataset from the Hub |
| 1 | **smoke run** of the training on the real files (a few batches, then the probes), **timing** of a step, of the real data loader and of a planning solve |
| 2 | **F1**, the default flat model: training (detached, resumes after a pod restart; 10 epochs are about 3.3 hours on an A100 at 13 it/s), loss curves, the probe suite of the final model |
| 3 | **F3**, the cheap P1 of F1 (task set A, offset 8) with the random baseline: the reference of every variant notebook, so run it (and upload) before them |
| 4 | **P2 and P3** (F4, F5): only for the final comparison of the flat model with the hierarchy, not now |
| 5 | runs and reports to the Hub |

Jobs are the ones of `experiments/experiments.yaml`, run by `experiments/run.py` (a job whose output exists is skipped). A step that finished is skipped when you run its cell again.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import os, shutil, sys, time
from pathlib import Path

REPO = Path("/root/AAC")                   # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True

SEEDS = "42"                               # comma-separated; headline results: "42,43,44"
MAX_EPOCHS = 10                            # 155k steps, 13 it/s on an A100 = 3.3 hours (2.8 in time_run.py); the cosine schedule ends here, it cannot be extended
USE_WANDB = True                           # live curves in Weights & Biases: the key is asked for below (or WANDB_API_KEY in the environment); False = the metrics CSV only
LOADER_BATCHES = 50                        # batches of the real loader timed in step 1
REPACK_PIXELS = False                      # step 1: rewrite the training file without compression if loading is the bottleneck (not needed on the A100 pod: 3500 clips/s loaded, 2000 needed)

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO, seeds=SEEDS)
from pod import DATA, H_JEPA, LOGS, PYTHON, RUNNER, launch, progress, run_dir, running, sh, show, wait

TRAIN_OVERRIDES = pod.train_overrides(USE_WANDB)   # Hydra overrides for every training run
api = pod.login(HF_REPO_ID, HF_PRIVATE, USE_WANDB)
pod.install(api, HF_REPO_ID, USE_WANDB)

## 1. Smoke run and timing

The **smoke run** is the real training command on the real files, a few batches and one validation pass, then the probe suite at the end (`probe_suite.py`: every entity along pixels, ViT tokens, CLS, latent, one step, rollout). It proves the columns, the paths and the reports on this pod before the long run. The **timing** measures a training step, the real data loader (the pixels are stored gzip-compressed in chunks of 100 frames, so loading can be the bottleneck) and a planning solve, and gives the GPU-hours for the configured epochs. `MAX_EPOCHS` is 10 (155k steps, 13 it/s on an A100 = 3.3 hours); change it only if the timing says otherwise.

In [ ]:
# --- 1. Smoke run on the real files --------------------------------------------------------------------------------------
# The full output is in LOGS/smoke.log. A traceback about environment.json in it is harmless: stable-pretraining sets file times on the volume,
# which it refuses, and carries on.
SMOKE = "flat_smoke"
shutil.rmtree(run_dir(SMOKE), ignore_errors=True)
t0 = time.time()
sh([PYTHON, "main_hjepa.py", "--config-name", "craftax_hjepa", "num_levels=1", f"output_model_name={SMOKE}", f"seed={SEEDS.split(',')[0]}", "trainer.max_epochs=1",
    "+trainer.limit_train_batches=20", "+trainer.limit_val_batches=2", "health.probe_clips=16", *TRAIN_OVERRIDES, "wandb.enabled=false"], cwd=H_JEPA, quiet=True, log="smoke")   # no W&B run for a smoke run
out = run_dir(SMOKE)
assert (out / f"{SMOKE}_object.ckpt").exists(), "the smoke run wrote no model"
assert (out / "health" / "report.md").exists() and (out / "health" / "probe_report.md").exists(), "the smoke run wrote no reports"
print(f"smoke run ok in {time.time() - t0:.0f} s: {out}")
show(out / "health" / "probe_report.md", 25)

In [ ]:
# --- 1. Timing: a training step, the real loader, a planning solve ---------------------------------------------------------
sh([PYTHON, "time_run.py", "--config-name", "craftax_hjepa", "--eval-config-name", "craftax_plan", "--loader-batches", LOADER_BATCHES,
    "num_levels=1", *TRAIN_OVERRIDES], cwd=H_JEPA)

In [ ]:
# --- 1. Optional: the training file without compression (only if the timing says loading is the bottleneck) ----------------
REPACK = """
import os, sys, h5py, numpy as np
path = sys.argv[1]
tmp = path + ".repack"
with h5py.File(path, "r") as src, h5py.File(tmp, "w") as dst:
    for key, value in src.attrs.items():
        dst.attrs[key] = value
    for name, ds in src.items():
        out = dst.create_dataset(name, shape=ds.shape, dtype=ds.dtype)   # contiguous, no compression
        step = 20_000
        for a in range(0, ds.shape[0], step):
            out[a : a + step] = ds[a : a + step]
        for key, value in ds.attrs.items():
            out.attrs[key] = value
    rng = np.random.default_rng(0)
    for name in src:
        rows = np.sort(rng.choice(src[name].shape[0], size=min(200, src[name].shape[0]), replace=False))
        assert np.array_equal(src[name][rows], dst[name][rows], equal_nan=src[name].dtype.kind == "f"), name
os.replace(tmp, path)
print("repacked", path)
"""
train_file = DATA / "craftax_expert_train.h5"
if REPACK_PIXELS:
    sh(["df", "-h", str(DATA)], env=os.environ)   # needs about 25 GB free
    sh([PYTHON, "-c", REPACK, train_file])
    sh([PYTHON, "time_run.py", "--config-name", "craftax_hjepa", "--steps", "3", "--loader-batches", LOADER_BATCHES, "num_levels=1", *TRAIN_OVERRIDES], cwd=H_JEPA)
else:
    print("REPACK_PIXELS is False: the training file stays as it is")

## 2. F1, the default flat model

`craftax_hjepa num_levels=1`: history 3, `rollout_n` 1, the CLS latent, IDM on. The job trains, runs the probe suite at the end of the training (`probe_suite.py`, `health.probe_clips` 512 clips per split, `probe_report.md`; 781 s on an A100 before the additions of 2026-10-27) and writes the comparison report. The separate probe job of `run.py` only runs if the training wrote no probe report (the probe failed); to probe again, delete `health/probe_report.md` and start the cell again. If the pod restarts, run the start cell again: the training resumes from `lightning_resume/last.ckpt`, the finished jobs are skipped.

Two epochs in a row with a critical finding stop the training (`health.stop_after_critical_epochs`). The model is still saved and the probe runs, so read `report.md` before planning. The **loss curves** cell draws the training curves and the health metrics from the run's metrics CSV (run it as often as you like). The probe suite runs on the final model only (settled on 2026-10-25). F1 was trained before the current suite: the **probe suite on F1** cell probes its final model again when its report is from an older suite (that report is kept in `health/probe_first_version/` or `health/probe_suite_v1/`). If this pod's volume does not have the run, the cell takes it from `runs/` of the Hub repo (where the last cell of this notebook uploads the runs). The suite runs detached like F1 (the cell returns at once); the cell after it shows its progress, the GPU, and the report when it is done.

In [ ]:
# --- 2. Start (or resume) F1, detached ---------------------------------------------------------------------------------------
assert MAX_EPOCHS, "set MAX_EPOCHS in the configuration cell from the timing"
OVERRIDES = [f"trainer.max_epochs={MAX_EPOCHS}", *TRAIN_OVERRIDES]
sh([*RUNNER, "plan", "F1", "--seeds", SEEDS, "--override", *OVERRIDES])
launch("F1", [*RUNNER, "run", "F1", "--mode", "local", "--seeds", SEEDS, "--override", *OVERRIDES])

In [ ]:
# --- 2. Monitor F1 (run as often as you like) ----------------------------------------------------------------------------------
print("F1", "RUNNING" if running("F1") else "not running")
print("\n".join(progress("F1", "Epoch", 2)))
print("\n".join(progress("F1", "[health]", 5)))
for p in sorted((run_dir("flat") / "health").glob("epoch_*.json"))[-1:]:
    print("last epoch report:", p.name)

In [ ]:
# --- 2. Loss curves of F1 (run as often as you like, also while it trains: the metrics CSV is written as it goes) ----------------------
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", run_dir("flat")])
display(Image(filename=str(run_dir("flat") / "health" / "loss_curves.png")))

In [ ]:
# --- 2. Wait for F1, then the training and probe reports ------------------------------------------------------------------------
wait("F1", "Epoch")
show(run_dir("flat") / "health" / "report.md", 40)
show(run_dir("flat") / "health" / "probe_report.md", 120)   # the probe suite's report once the training wrote it

In [ ]:
# --- 2. The probe suite on F1's final model, when its report is from an older suite (F1 was trained before it) ---------
# The run is taken from the Hub (runs/, uploaded by the last cell of this notebook) when this pod's volume does not have it.
# 781 s on an A100 before the additions of 2026-10-27; the older report is kept (pod.current_probe_report). It runs detached;
# the log is LOGS/probe_suite.log (the next cell follows it).
run = run_dir("flat")
if not (run / "flat_object.ckpt").exists():
    pod.fetch(api, HF_REPO_ID, f"flat/seed{SEEDS.split(',')[0]}")
if pod.current_probe_report(run):
    print(f"{run / 'health' / 'probe_report.md'} is the current suite's: delete it to run the suite again")
else:   # detached like F1, so the cell returns at once and the log is written as it goes: follow it in the next cell
    launch("probe_suite", [PYTHON, "probe_suite.py", "--run", run, "--checkpoint", run / "flat_object.ckpt"], cwd=H_JEPA)

In [ ]:
# --- 2. Monitor the probe suite (run as often as you like; it writes health/probe_report.md when it is done) ---------------
print("probe suite", "RUNNING" if running("probe_suite") else "not running")
print("\n".join(progress("probe_suite", "[probe suite]", 3)))   # one line per finished stage; the long stages print nothing in between
sh(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"], env=os.environ)   # busy GPU: it works, not stalled
report = run_dir("flat") / "health" / "probe_report.md"
if not running("probe_suite"):
    if report.exists():
        show(report, 200)
    else:
        print("no report: the end of the log follows")
        print("".join((LOGS / "probe_suite.log").read_text(errors="replace").splitlines(keepends=True)[-60:]))

## 3. F3: the cheap P1 of F1, the reference of the variants

Task set A of `experiments.yaml` (settled on 2026-10-27): 300 fixed-offset goals at offset 8 on the validation worlds (planning horizon 8), a budget of 16 steps, the random baseline on the same tasks, GIFs of the first 8 tasks (`videos/`). Every variant notebook compares its run with these results: run this step and the upload (step 5) before the variant notebooks. The variant notebooks plan F1 themselves when they find no results on the Hub (the same job, skipped when done).

In [ ]:
# --- 3. Start F3 (F1 and the random baseline on task set A), detached ------------------------------------------------------------
sh([*RUNNER, "plan", "F3", "--seeds", SEEDS])
launch("F3", [*RUNNER, "run", "F3", "--mode", "local", "--seeds", SEEDS])

In [ ]:
# --- 3. Monitor and read F3 (run as often as you like) ----------------------------------------------------------------------------
print("F3", "RUNNING" if running("F3") else "not running")
print("\n".join(progress("F3", "success_rate", 3)))
for metrics in sorted((DATA / "ckpts" / "craftax").glob("*/**/eval_A_*/metrics.yaml")):
    rate = next((l for l in metrics.read_text().splitlines() if l.startswith("success_rate")), "")
    videos = len(list((metrics.parent / "videos").glob("*.gif")))
    print(f"{metrics.parent.relative_to(DATA)}: {rate}, {videos} videos")
for report in sorted((DATA / "reports").glob("F3_*.md")):
    print("\n=====", report.name)
    show(report, 30)

## 4. P2 and P3 (F4, F5): for the final comparison only

Settled on 2026-10-27: the flat variants are planned on the cheap set A only; P2 (F4, the diamond through oracle subgoals, task set C) and P3 (F5, the diamond from the final goal only, task set B) are kept for the final comparison of the flat model with the hierarchy. **F2 is not designed yet**: the planning settings for P2 and P3 are chosen on the final flat model, put into `experiments/experiments.yaml` and `H-JEPA/h_jepa/config/eval/craftax_plan.yaml`, committed and pushed, then pulled here (the next cell does). Reset the notebook on the pod before it (`git checkout notebooks/runpod_flat.ipynb`) if you edited it there: a pull stops at local changes to a file that changed upstream.

In [ ]:
# --- 4. Pull the frozen planning settings, then start P2 and P3 one after the other, detached ---------------------------------------
sh(["git", "-C", REPO, "pull", "--ff-only"], env=os.environ)
sh(["git", "-C", REPO, "log", "-1", "--oneline"], env=os.environ)
CHAIN = """
import subprocess, sys
runner, seeds = sys.argv[1:3]
for exp in sys.argv[3:]:
    print("=====", exp, flush=True)
    code = subprocess.call([sys.executable, runner, "run", exp, "--mode", "local", "--seeds", seeds])
    if code:
        sys.exit(f"{exp} failed with exit code {code}")
"""
launch("planning", [PYTHON, "-c", CHAIN, RUNNER[1], SEEDS, "F4", "F5"])

In [ ]:
# --- 4. Monitor and read P2 and P3 (run as often as you like) ---------------------------------------------------------------------------
print("planning", "RUNNING" if running("planning") else "not running")
print("\n".join(progress("planning", "=====", 1) + progress("planning", "success_rate", 3)))
for metrics in sorted((DATA / "ckpts" / "craftax").glob("*/**/eval_[BC]_*/metrics.yaml")):
    rate = next((l for l in metrics.read_text().splitlines() if l.startswith("success_rate")), "")
    videos = len(list((metrics.parent / "videos").glob("*.gif")))
    print(f"{metrics.parent.relative_to(DATA)}: {rate}, {videos} videos")
for report in sorted((DATA / "reports").glob("F[45]_*.md")):
    print("\n=====", report.name)
    show(report, 30)

## 5. Upload

The runs (final models, configs, health and probe reports, planning results with their videos) and the comparison reports go to `runs/` and `reports/` of the same Hub repo, where the variant notebooks take F1 and its F3 results from. Left out: the resumable training state (`lightning_resume/`), the per-epoch models and the smoke runs.

In [ ]:
# --- 5. Runs and reports to the Hub --------------------------------------------------------------------------------------------------------
pod.upload(api, HF_REPO_ID)   # every run of ckpts/craftax, the F and V reports (pod.UPLOAD_IGNORE: lightning_resume/, *_epoch_*_object.ckpt, *_smoke/)